In [43]:
# Проерка версий библиотек
# Версия Python
import sys
print('Python: {}'.format(sys.version))

# Загрузка scipy
import scipy
print('scipy: {}'.format(scipy.__version__))

# Загрузка numpy
import numpy
print('numpy: {}'.format(numpy.__version__))

# Загрузка matplotlib
import matplotlib
print('matplotlib: {}'.format(matplotlib.__version__))

# Загрузка pandas
import pandas
print('pandas: {}'.format(pandas.__version__))

# Загрукзка scikit-learn
import sklearn
print('sklearn: {}'.format(sklearn.__version__))

Python: 3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]
scipy: 1.10.1
numpy: 1.24.3
matplotlib: 3.7.1
pandas: 2.0.1
sklearn: 1.2.2


Импортируем необходимые модули и сверяем версии

In [44]:
# Загрузка библиотек
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns
from fitter import Fitter
import warnings
import time
from pandas import read_csv
from pandas.plotting import scatter_matrix
from matplotlib import pyplot
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from collections import Counter

# Считывание файлов

In [ ]:
def readFileTXT(filename):
    # Считывание файла формата TXT
    df_ = pd.read_table(filename, delimiter = ',')
    print(df_)
    return(df_)

In [ ]:
def readFileCSV(filename):
    # Считывание файла формата CSV
    df_ = pd.read_csv(filename,) 
    print(df_)
    return(df_)

In [ ]:
def readFileXSLX(filename):
    # Считывание файла формата XLSX 
    df_ = pd.read_excel(filename)  
    print(df_)
    return(df_)

In [ ]:
def readFileJSON(filename):
    # Считывание файла формата JSON 
    df_ = pd.read_json(filename)  
    print(df_)
    return(df_)

# Работа с параметрами

In [ ]:
def MissingValues(df):
    mis_val = df.isnull().sum()
        
    # Процент пустых значений по всем записям и всем параметрам
    mis_val_percent = 100 * df.isnull().sum() / len(df)

    # Сводная таблица по пустым значениям параметров
    mis_val_table = pd.concat([mis_val, mis_val_percent], axis=1)

    # Переименовка колонок сводной таблицы
    mis_val_table_ren_columns = mis_val_table.rename(
    columns = {0 : 'Сколько пропущенных значений', 1 : '% от общего количества значений'})

    # Сортируем по убыванию процента пропущенных значений
    mis_val_table_ren_columns = mis_val_table_ren_columns[
        mis_val_table_ren_columns.iloc[:,1] != 0].sort_values(
    '% от общего количества значений', ascending=False).round(1)

    # Выводим информацию для итога
    print ("В выборке данных " + str(df.shape[1]) + " столбцов.\n"      
        "В ней " + str(mis_val_table_ren_columns.shape[0]) +
          " столбцов, в которых есть нулевые данные.")
    return(mis_val_table_ren_columns)

In [ ]:
def DelRowMissValPerc(df, mis_val_table_ren_columns):
    missing_df = mis_val_table_ren_columns
    missing_columns = list(missing_df[missing_df['% от общего количества значений'] > 50].index)
    print('Удаляем %d столбцов.' % len(missing_columns))
    df = df.drop(columns = list(missing_columns))
    df.describe()

In [ ]:
def countParametersInTotal(df, ids, folder, fileformat):
    x = Counter()
    for id in ids:
        df_ = pd.read_table(folder+"/" + str(id) + fileformat)
        # Поменять функцию если другой тип данных
        x = x + Counter(df_["Parameter"])
    return(x)

In [ ]:
def devideParameters(x, df):
    # Разделение параметров по встречаемости
    unique_parameters = list(x.keys())
    n = len(df.index)
    one_params = []
    mean_params = []
    rare_params = []
    median_params = []
    for parameter in unique_parameters:
        if x[parameter] / n > 10:
            median_params.append(parameter)
        elif x[parameter] / n > 1:
            mean_params.append(parameter)
        elif x[parameter] / n < 1:
            rare_params.append(parameter)
        else:
            one_params.append(parameter)

    feature_list = one_params + rare_params + mean_params + median_params
    return(feature_list, one_params, rare_params, mean_params, median_params)

In [ ]:
def addMissingParameters(df, df_ids, feature_list, one_params, rare_params, mean_params, median_params):
    df = pd.DataFrame(columns = feature_list, index = range(n))
    for i, rid in enumerate(df_ids):
        df_ = pd.read_table('set/' + str(rid) + '.txt', delimiter = ',')
        df_edited = pd.DataFrame(0, index = range(1), columns = feature_list)
        for parameter in one_params:
            df_edited[parameter] = df_.loc[df_["Parameter"] == parameter, 'Value'].sum()
        # Считаем значения единичных параметров в одной записи ( сумм просто значение этого параметра вводит в запись )
        for parameter in rare_params:
            df_edited[parameter] = df_.loc[df_["Parameter"] == parameter, 'Value'].mean()
        # Считываем показания ( для одного пациента несколько раз встречается один и тот же параметр ) и берем СРЕДНЕЕ ЗНАЧЕНИЕ для редких
        for parameter in mean_params:
            df_edited[parameter] = df_.loc[df_["Parameter"] == parameter, 'Value'].mean()
        # Считываем показания ( для одного пациента несколько раз встречается один и тот же параметр ) и берем СРЕДНЕЕ ЗНАЧЕНИЕ для средних
        for parameter in median_params: 
            df_edited[parameter] = df_.loc[df_["Parameter"] == parameter, 'Value'].median()
        # Считываем показания ( для одного пациента несколько раз встречается один и тот же параметр ) и берем Медиану для 
        # параметров с большим разбросом
        df.loc[i, feature_list] = df_edited.iloc[0].values
        # Вписываем вычисленные значения для единичной записи пациента в датасете
    return(df)

In [ ]:
def createCSVDataset(df):
    out = pd.read_table('outcomes.txt', delimiter = ',')
    # Считываем уникальных пациентов и результаты (с помощью результатов будем прогнозировать смерть)
    df["Survival"] = out["Survival"]
    # Вставляем для каждого пациента свои результаты
    df["In-hospital_death"] = out["In-hospital_death"]
    # Вставляем для каждого пациента свои результаты
    print(df)
    df.to_csv('dataset.csv', index=False)
    # По сути мы приписали правые столбцы для каждой записи

In [ ]:
def describeData(filename):
    df = pd.read_csv(filename)
    df.describe(include= "all")
    # Названия полей
    df.columns
    # типы полей
    df.info()
    

In [52]:
def describeParams(rare_params, mean_params, one_params, median_params):
    # Для полей из rare-params и mean-params рассчитывается среднее значение из имеющихся по человеку 
    print("rare_params: ", rare_params)
    print("mean_params: ", mean_params)
    # Для полей из one-params Выдаётся единственное имеющееся значение
    print("one_params: ", one_params)
    # Для полей из median_params рассчитывается медианное значение
    print("median_params: ", median_params)

rare_params:  ['Albumin', 'ALP', 'ALT', 'AST', 'Bilirubin', 'Cholesterol', 'TroponinI', 'TroponinT']
mean_params:  ['HCT', 'BUN', 'Creatinine', 'Glucose', 'HCO3', 'Mg', 'Platelets', 'K', 'Na', 'WBC', 'pH', 'PaCO2', 'PaO2', 'FiO2', 'MechVent', 'SaO2', 'Lactate']
one_params:  ['RecordID', 'Age', 'Gender', 'Height', 'ICUType']
median_params:  ['Weight', 'GCS', 'HR', 'NIDiasABP', 'NIMAP', 'NISysABP', 'RespRate', 'Temp', 'Urine', 'DiasABP', 'MAP', 'SysABP']


In [ ]:
def duplicateCheck(filename):
    df = pd.read_csv(filename)
    df.info()
    print("Дубликаты: ",df.duplicated().any())

#### Дубликатов нет, действий не требуется

In [57]:
def removeDuplicates(df):
    df.dropna(how='all')

,RecordID,Age,Gender,Height,ICUType,Albumin,ALP,ALT,AST,Bilirubin,...,NIMAP,NISysABP,RespRate,Temp,Urine,DiasABP,MAP,SysABP,Survival,In-hospital_death
0,132539.0,54.0,0.0,-1.0,4.0,NaN,NaN,NaN,NaN,NaN,...,70.000,110.0,18.0,37.70,100.0,NaN,NaN,NaN,-1,0
1,132540.0,76.0,1.0,175.3,2.0,NaN,NaN,NaN,NaN,NaN,...,78.165,115.0,NaN,37.45,90.0,59.0,79.0,116.5,-1,0
2,132541.0,44.0,0.0,-1.0,3.0,2.5,116.0,83.0,199.5,2.9,...,97.670,134.0,NaN,37.85,100.0,67.0,90.0,125.0,-1,0
3,132543.0,68.0,1.0,180.3,3.0,4.4,105.0,12.0,15.0,0.2,...,83.670,120.0,16.0,36.40,625.0,NaN,NaN,NaN,575,0
4,132545.0,88.0,0.0,-1.0,3.0,3.3,NaN,NaN,NaN,NaN,...,75.330,131.0,19.0,37.00,50.0,NaN,NaN,NaN,918,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7995,152849.0,78.0,1.0,180.3,2.0,NaN,NaN,NaN,NaN,NaN,...,83.330,143.0,NaN,36.85,80.0,62.0,83.0,130.0,752,0
7996,152851.0,90.0,1.0,177.8,3.0,NaN,74.0,12.0,25.0,1.0,...,63.835,108.5,NaN,37.15,20.0,46.5,71.0,120.5,39,0
7997,152858.0,70.0,0.0,152.4,2.0,2.8,88.0,11.0,21.0,NaN,...,90.670,143.0,17.0,36.60,115.0,60.5,82.5,123.5,334,0
7998,152862.0,49.0,0.0,-1.0,3.0,NaN,NaN,NaN,NaN,NaN,...,100.500,137.5,NaN,36.20,325.0,NaN,NaN,NaN,-1,0


In [ ]:
def RSSparametersCheck(df):
    i = 0
    f = [None] * (df.dtypes=='float64').sum()
    for column in df.columns:
        if df.dtypes[column]=='float64':
            f[i] = Fitter(df[column][~df[column].isnull()],
                       distributions=["expon", "uniform", "norm", "lognorm"],
                       bins = 15)
            time.sleep(0.2)
            f[i].fit()
            print(column)
            print(f[i].summary()['sumsquare_error'])
            i += 1  
    return(f)

In [ ]:
def ParameterSummary(f,n):
    f[n].summary()

In [ ]:
def AnomalyDetecter(df):
    for column in df.columns:
        if column not in ["здесь пишем колонки"] and df.dtypes[column] != 'bool':
            # # Вычисляем первый и третий квартиль (он же квантиль)
            # value = df[column]
            # q1, q3 = value.quantile([0.25, 0.75])

            # # Межквартильный размах, допустимых значений
            # iq = q3 - q1
            # low = q1 - 1.5*iq
            # high = q3 + 1.5*iq

            # # Удаляем значения вне межквартильного размаха (т.к. это аномалии)
            # if value < low or value> high:
            #     value = np.nan
            values = df[column] 
            q1, q3 = values.quantile([0.25, 0.75])
            low  = q1 - (q3 - q1) * 5
            high = q3 + (q3 - q1) * 5
            condition = (values < low) | (values > high)
            df[column][condition] = np.nan
    print(df)

In [ ]:
def AverageVal(df):
    for column in df.columns:
        if column not in ["Айдизаписи"]:
            values = df[column] 
            E = values.mean()
            D = values.var()
            sigma = np.log(D / (E ** 2) + 1) ** 0.5
            mu = np.log(E) - (sigma ** 2) / 2
            condition = values.isna()
            new_values =  abs(np.random.lognormal(mu, sigma, len(df.index)))
            values[condition] = new_values[condition]


In [ ]:
df.isna().sum().sum()
# Проверка на миссинги

In [ ]:
def CheckNan(df, col):
    # Пример-проверка, что NaN удалились в столбце Lactate ( там было 45% NaN)
    data = df.to_numpy()
    figure, axis = plt.subplots(1, 1)
    figure.set_size_inches(15, 5)
    s = pd.Series(data[:, df.columns.get_loc(col)])
    print(len(s), len(df.index))
    axis.hist(s, bins = 10, log = True)
    axis.set_title(col)
    plt.show()

In [ ]:
df.to_csv('dataset_2.csv', index = False)

# Работа с графиками

In [ ]:
def PlotMissingValue(df):
    plt.plot((df.isnull().sum(axis = 1)).sort_values().values)

In [ ]:
def PlotColDataBox(df):
    for column in df.columns:
        plt.figure()
        df.boxplot([column])
    plt.show()

In [ ]:
def ColumnsPlots(df):
    data = df.to_numpy()
    figure, axis = plt.subplots((df.dtypes=='float64').sum(), 1)
    figure.set_size_inches(15, 250)
    i = 0
    for column in df.columns:
        if df.dtypes[column]=='float64':
            s = pd.Series(data[:, df.columns.get_loc(column)])
            axis[i].hist(s, bins = 10, log = True)
            axis[i].set_title(column)
            i += 1
    plt.show()

In [ ]:
def HeatMapCorr(df):
    sns.heatmap(df.corr())